# Modern Data Engineering for AI Systems — Colab Demo

This lightweight notebook demonstrates the project's core data-modeling workflow with the committed synthetic events. It is an **educational companion**, not a replacement for the production Docker architecture. Kafka, Spark, Delta Lake, Airflow, Great Expectations, OpenLineage, OpenSearch, and Ollama remain the production implementation.

## Colab-to-production mapping

| Colab demonstration | Production implementation |
|---|---|
| JSON fixture loading | Kafka ingestion |
| Real Pydantic `CustomerEvent` validation | Kafka boundary validation |
| In-memory quarantine DataFrame | Kafka `quarantine.events` topic |
| Pandas Bronze DataFrame | Bronze Delta table with Kafka provenance |
| Pandas deterministic deduplication | PySpark plus Delta `MERGE` |
| Pandas latest-state selection | Current-state Silver Delta table |
| Pandas grouped metrics | Currency-aware Gold Delta table |
| Notebook assertions | Great Expectations gates and Airflow dependencies |

**Dataset grains**

- Historical Silver: one row per `event_id`
- Current-state Silver: one row per `(customer_id, event_type)`
- Gold: one row per `(customer_id, event_day, currency)`

In [ ]:
# Lightweight setup: Colab already includes pandas and matplotlib.
from pathlib import Path
import importlib.util
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Norahwalled/Engineering-capstone.git"
COLAB_ROOT = Path("/content/Engineering-capstone")

if not importlib.util.find_spec("pydantic"):
    subprocess.run([sys.executable, "-m", "pip", "install", "pydantic==2.10.6"], check=True)

if (Path.cwd() / "src" / "capstone_de").exists():
    PROJECT_ROOT = Path.cwd()
else:
    if not COLAB_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(COLAB_ROOT)], check=True)
    PROJECT_ROOT = COLAB_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
from datetime import UTC, datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd
from pydantic import ValidationError

from capstone_de.domain.events import CustomerEvent

SAMPLE_ROOT = PROJECT_ROOT / "data" / "samples"
FILES = {
    "valid": "valid_events.json",
    "duplicates": "duplicate_events.json",
    "late": "late_events.json",
    "invalid": "invalid_events.json",
}
batches = {name: json.loads((SAMPLE_ROOT / filename).read_text()) for name, filename in FILES.items()}
pd.DataFrame({"batch": batches.keys(), "records": [len(records) for records in batches.values()]})

## Load raw deliveries

Publication order models a normal batch, replayed deliveries, older events arriving late, and invalid records. Synthetic Kafka coordinates and ingestion timestamps make provenance and deduplication visible without running Kafka.

In [ ]:
raw_deliveries = []
base_ingestion_time = datetime(2026, 8, 6, 8, 0, tzinfo=UTC)
delivery_order = 0
for batch_order, batch_name in enumerate(("valid", "duplicates", "late", "invalid"), start=1):
    for payload in batches[batch_name]:
        raw_deliveries.append({
            "payload": payload,
            "source_file": FILES[batch_name],
            "batch_order": batch_order,
            "delivery_order": delivery_order,
            "kafka_partition": delivery_order % 3,
            "kafka_offset": delivery_order,
            "kafka_timestamp": base_ingestion_time + timedelta(seconds=delivery_order),
            "ingested_at": base_ingestion_time + timedelta(seconds=delivery_order),
        })
        delivery_order += 1

assert len(raw_deliveries) == 18
pd.DataFrame([{"source_file": row["source_file"], **row["payload"]} for row in raw_deliveries]).head()

## Validate events and demonstrate quarantine

The notebook imports the repository's real `CustomerEvent` contract. Valid records continue to Bronze; rejected payloads retain their source coordinates and validation reasons, mirroring the production quarantine envelope. Duplicate deliveries may be contract-valid—the Silver layer handles idempotency.

In [ ]:
validated_deliveries = []
quarantine_records = []
for delivery in raw_deliveries:
    try:
        event = CustomerEvent.model_validate(delivery["payload"])
        normalized = event.model_dump(mode="python")
        normalized["event_id"] = str(normalized["event_id"])
        normalized["amount"] = float(normalized["amount"])
        validated_deliveries.append({**normalized, **{key: delivery[key] for key in delivery if key != "payload"}})
    except ValidationError as error:
        quarantine_records.append({
            "event_reference": delivery["payload"].get("event_id", "missing"),
            "source_file": delivery["source_file"],
            "kafka_partition": delivery["kafka_partition"],
            "kafka_offset": delivery["kafka_offset"],
            "invalid_fields": ", ".join(sorted({str(item["loc"][0]) for item in error.errors()})),
            "rejection_reason": "; ".join(item["msg"] for item in error.errors()),
        })

quarantine_df = pd.DataFrame(quarantine_records)
assert len(validated_deliveries) == 14
assert len(quarantine_df) == 4
quarantine_df[["event_reference", "invalid_fields", "kafka_offset"]]

## Bronze simulation

Bronze preserves every valid source delivery and its provenance. Replayed IDs are retained intentionally.

In [ ]:
bronze_df = pd.DataFrame(validated_deliveries)
bronze_summary = pd.Series({
    "bronze_deliveries": len(bronze_df),
    "unique_event_ids": bronze_df["event_id"].nunique(),
    "replayed_deliveries": len(bronze_df) - bronze_df["event_id"].nunique(),
})
assert bronze_summary.to_dict() == {"bronze_deliveries": 14, "unique_event_ids": 11, "replayed_deliveries": 3}
bronze_summary.to_frame("rows")

## Historical and current-state Silver simulations

Historical Silver deterministically retains the newest delivery for each immutable `event_id`. Current-state Silver is a separate derived view: it selects the newest business event for each `(customer_id, event_type)` without deleting history.

In [ ]:
delivery_order_columns = ["event_id", "ingested_at", "kafka_timestamp", "kafka_partition", "kafka_offset"]
silver_history_df = (
    bronze_df.sort_values(delivery_order_columns)
    .drop_duplicates(subset=["event_id"], keep="last")
    .sort_values(["occurred_at", "event_id"])
    .reset_index(drop=True)
)
assert len(silver_history_df) == 11
assert silver_history_df["event_id"].is_unique
assert (silver_history_df["amount"] >= 0).all()

current_order = ["customer_id", "event_type", "occurred_at", "ingested_at", "kafka_timestamp", "kafka_partition", "kafka_offset", "event_id"]
silver_current_df = (
    silver_history_df.sort_values(current_order)
    .drop_duplicates(subset=["customer_id", "event_type"], keep="last")
    .sort_values(["customer_id", "event_type"])
    .reset_index(drop=True)
)
assert not silver_current_df.duplicated(["customer_id", "event_type"]).any()
print(f"Historical Silver: {len(silver_history_df)} rows; current-state Silver: {len(silver_current_df)} rows")
silver_current_df[["customer_id", "event_type", "event_id", "occurred_at"]]

In [ ]:
# Late events remain attached to their original event dates and do not displace newer state.
late_ids = {record["event_id"] for record in batches["late"]}
late_in_history = silver_history_df[silver_history_df["event_id"].isin(late_ids)][
    ["event_id", "customer_id", "event_type", "occurred_at", "ingested_at"]
]
assert len(late_in_history) == 3
late_in_history

## Currency-aware Gold metrics

Gold is built from the complete historical Silver table—not the current-state view. Including `currency` in the grain prevents financially meaningless sums across SAR, USD, and EUR.

In [ ]:
gold_input = silver_history_df.copy()
gold_input["event_day"] = pd.to_datetime(gold_input["occurred_at"], utc=True).dt.floor("D")
gold_df = (
    gold_input.groupby(["customer_id", "event_day", "currency"], as_index=False)
    .agg(event_count=("event_id", "count"), total_amount=("amount", "sum"), average_amount=("amount", "mean"))
    .sort_values(["event_day", "customer_id", "currency"])
    .reset_index(drop=True)
)
assert not gold_df.duplicated(["customer_id", "event_day", "currency"]).any()
assert gold_df["currency"].notna().all()
assert gold_df["event_count"].sum() == len(silver_history_df) == 11
gold_df

## Explicit multi-currency safety check

This controlled, in-memory extension proves that two currencies for the same customer and day create separate Gold rows. It does not change the committed fixtures or their reconciliation counts.

In [ ]:
currency_safety_input = pd.DataFrame([
    {"customer_id": "demo-customer", "event_day": pd.Timestamp("2026-08-04", tz="UTC"), "currency": "SAR", "event_id": "demo-sar", "amount": 100.00},
    {"customer_id": "demo-customer", "event_day": pd.Timestamp("2026-08-04", tz="UTC"), "currency": "USD", "event_id": "demo-usd", "amount": 25.00},
])
currency_safety_gold = currency_safety_input.groupby(
    ["customer_id", "event_day", "currency"], as_index=False
).agg(event_count=("event_id", "count"), total_amount=("amount", "sum"))
assert len(currency_safety_gold) == 2
assert set(zip(currency_safety_gold["currency"], currency_safety_gold["total_amount"])) == {("SAR", 100.0), ("USD", 25.0)}
currency_safety_gold

## Pipeline reconciliation

In [ ]:
reconciliation = pd.DataFrame({
    "stage": ["Raw deliveries", "Validated deliveries", "Quarantine", "Bronze", "Historical Silver", "Current-state Silver", "Gold event-count total"],
    "rows": [18, 14, 4, len(bronze_df), len(silver_history_df), len(silver_current_df), int(gold_df["event_count"].sum())],
})
assert reconciliation.set_index("stage").loc["Gold event-count total", "rows"] == 11
reconciliation

## Visual results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

funnel = reconciliation[reconciliation["stage"].isin(["Raw deliveries", "Validated deliveries", "Bronze", "Historical Silver"])]
axes[0].bar(funnel["stage"], funnel["rows"], color=["#4C78A8", "#59A14F", "#76B7B2", "#F28E2B"])
axes[0].set_title("Pipeline row reconciliation")
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_ylabel("Rows")

event_counts = silver_history_df["event_type"].value_counts().sort_values()
axes[1].barh(event_counts.index, event_counts.values, color="#59A14F")
axes[1].set_title("Historical events by type")
axes[1].set_xlabel("Events")

for currency, group in gold_df.groupby("currency"):
    daily = group.groupby("event_day")["total_amount"].sum()
    axes[2].plot(daily.index, daily.values, marker="o", label=currency)
axes[2].set_title("Daily totals shown separately by currency")
axes[2].set_ylabel("Amount (not combined across currencies)")
axes[2].legend(title="Currency")
axes[2].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

## Production disclaimer

This notebook proves the business rules on a tiny synthetic dataset; it does **not** execute or replace the production platform. The repository's production workflow uses Kafka for ingestion and quarantine, PySpark and Delta Lake for Bronze/Silver/Gold persistence, Great Expectations for blocking quality gates, Airflow for orchestration, OpenLineage and Marquez for lineage, and OpenSearch plus the RAG API downstream.

For the real architecture and verification process, see `README.md`, `docs/architecture.md`, `docs/verification.md`, and `data/README.md`.